# 03 — AI Use Cases

**Learning objectives**
- Understand what Computer Vision is and how image classifiers work
- Apply transfer learning to train a binary classifier on real images
- Evaluate the model with the metrics from notebook 02
- Build intuition for the gap between "training" and "generalising"

**Prerequisites:** Notebooks 01 & 02

---

## 1. Computer Vision

Computer Vision is the field of teaching machines to interpret images and video. It was one of the first major breakthroughs in modern AI, and it's the reason deep learning became dominant.

The core task: given an image (a grid of pixels), produce a label, a bounding box, a segmentation mask, or a generated output.

### Why is it hard?

A 224×224 colour image is a tensor of 224 × 224 × 3 = **150,528 numbers**. Two photos of the same husky can look completely different depending on lighting, angle, distance, and background. A hand-coded rule system has no chance.

Deep learning solves this by learning hierarchical features from raw pixels:

```
Layer 1:  edges and gradients
Layer 2:  corners, curves, textures
Layer 3:  eyes, ears, snouts
Layer 4:  "this is a husky"
```

No human wrote those rules. The network discovered them from labelled examples.

### What we're building

A **binary classifier**: given a photo, output the probability that it contains a **Siberian Husky**.

- Dataset: [Stanford Dogs / ImageNet Dogs](http://vision.stanford.edu/aditya86/ImageNetDogs/) — 120 breeds, ~20,000 images
- Positive class: Siberian Husky (~150 images)
- Negative class: equal sample from all other breeds
- Model: pretrained **ResNet-18** with a custom binary output head (transfer learning)

## 2. Transfer Learning

Training a deep CNN from scratch requires millions of images and days of GPU time. We don't have that.

**Transfer learning** sidesteps this: take a model already trained on a massive dataset (ImageNet — 1.2M images, 1000 classes), and repurpose it for your specific task.

The intuition: the early layers of a vision model learn universal features (edges, textures, shapes) that are useful for *any* image task. Only the final classification layer is task-specific.

**Strategy:**
1. Load ResNet-18 pretrained on ImageNet
2. **Freeze** all layers except the last — their weights won't change
3. Replace the final layer with a new one: `Linear(512 → 1)` for binary output
4. Train only that new layer on our husky data

This means we're training ~513 parameters instead of ~11 million. Fast, and it works remarkably well on small datasets.

In [ ]:
import urllib.request
import tarfile
import os
from pathlib import Path
from tqdm import tqdm

DATA_DIR  = Path("data/stanford_dogs")
IMAGE_DIR = DATA_DIR / "Images"
URL       = "http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar"
TAR_PATH  = DATA_DIR / "images.tar"

DATA_DIR.mkdir(parents=True, exist_ok=True)

class DownloadProgress(tqdm):
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize is not None:
            self.total = tsize
        self.update(b * bsize - self.n)

if IMAGE_DIR.exists():
    print(f"Dataset already extracted at {IMAGE_DIR}")
else:
    if not TAR_PATH.exists():
        print("Downloading Stanford Dogs dataset (~750 MB) ...")
        with DownloadProgress(unit='B', unit_scale=True, miniters=1, desc="images.tar") as t:
            urllib.request.urlretrieve(URL, TAR_PATH, reporthook=t.update_to)
        print("Download complete.")
    else:
        print("Tar file already downloaded, extracting ...")

    print("Extracting ...")
    with tarfile.open(TAR_PATH) as tar:
        tar.extractall(DATA_DIR)
    print("Done.")

breeds = sorted([d.name for d in IMAGE_DIR.iterdir() if d.is_dir()])
print(f"\n{len(breeds)} breeds found.")
husky_breeds = [b for b in breeds if "husky" in b.lower() or "Husky" in b]
print(f"Husky breed folders: {husky_breeds}")

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

random.seed(42)

HUSKY_FOLDER = husky_breeds[0]   # e.g. "n02110185-Siberian_husky"

# Collect husky images (positive class)
husky_images = sorted((IMAGE_DIR / HUSKY_FOLDER).glob("*.jpg"))

# Collect non-husky images (negative class) — balanced sample across other breeds
other_breeds  = [b for b in breeds if b != HUSKY_FOLDER]
n_per_breed   = max(1, len(husky_images) // len(other_breeds) + 1)
other_images  = []
for b in other_breeds:
    imgs = sorted((IMAGE_DIR / b).glob("*.jpg"))
    other_images.extend(random.sample(imgs, min(n_per_breed, len(imgs))))

# Trim to same count as huskies
random.shuffle(other_images)
other_images = other_images[:len(husky_images)]

print(f"Husky images:     {len(husky_images)}")
print(f"Non-husky images: {len(other_images)}")
print(f"Total:            {len(husky_images) + len(other_images)}")

# Preview a few from each class
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle("Sample Images — Top row: Husky  |  Bottom row: Not Husky", fontsize=13)
for i, ax in enumerate(axes[0]):
    ax.imshow(Image.open(husky_images[i]))
    ax.axis("off"); ax.set_title("Husky ✓", fontsize=9, color="green")
for i, ax in enumerate(axes[1]):
    ax.imshow(Image.open(other_images[i]))
    ax.axis("off"); ax.set_title("Not Husky ✗", fontsize=9, color="gray")
plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split

# ── Device ────────────────────────────────────────────────────────────────────
device = (
    torch.device("mps")  if torch.backends.mps.is_available() else
    torch.device("cuda") if torch.cuda.is_available()         else
    torch.device("cpu")
)
print(f"Using device: {device}")

# ── Transforms ────────────────────────────────────────────────────────────────
# ResNet-18 expects 224×224, normalised to ImageNet mean/std
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

# ── Dataset ───────────────────────────────────────────────────────────────────
class HuskyDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths     = paths
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(self.labels[idx], dtype=torch.float32)

# Build path/label lists and split 80/20
all_paths  = husky_images + other_images
all_labels = [1] * len(husky_images) + [0] * len(other_images)

train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels, test_size=0.2, random_state=42, stratify=all_labels
)

train_ds = HuskyDataset(train_paths, train_labels, train_transform)
val_ds   = HuskyDataset(val_paths,   val_labels,   val_transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0)

print(f"Train: {len(train_ds)} images  |  Val: {len(val_ds)} images")
print(f"Batches per epoch: {len(train_loader)} train, {len(val_loader)} val")

In [ ]:
# ── Model: ResNet-18 with frozen backbone ─────────────────────────────────────
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Freeze all layers — we don't want to destroy ImageNet features
for param in model.parameters():
    param.requires_grad = False

# Replace the final classification layer (512 → 1 for binary output)
model.fc = nn.Linear(model.fc.in_features, 1)
# Only the new layer has requires_grad=True

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Total parameters:     {total:,}")
print(f"Trainable parameters: {trainable:,}  ({100*trainable/total:.2f}%)")
print(f"\nModel head: {model.fc}")
print("\nAll other layers are frozen — only the final Linear layer will be updated.")

In [ ]:
# ── Training ──────────────────────────────────────────────────────────────────
criterion = nn.BCEWithLogitsLoss()   # sigmoid + binary cross-entropy, numerically stable
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

EPOCHS = 15

history = {"train_loss": [], "val_loss": [], "val_acc": []}

for epoch in range(EPOCHS):
    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs).squeeze(1)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(imgs)
    train_loss = running_loss / len(train_ds)

    # ── Validate ──────────────────────────────────────────────────────────────
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs).squeeze(1)
            val_loss += criterion(logits, labels).item() * len(imgs)
            preds    = (torch.sigmoid(logits) >= 0.5).float()
            correct  += (preds == labels).sum().item()
            total    += len(labels)

    val_loss /= len(val_ds)
    val_acc   = correct / total

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    scheduler.step()

    print(f"Epoch {epoch+1:02d}/{EPOCHS}  "
          f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.3f}")

# ── Training curves ───────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
epochs = range(1, EPOCHS + 1)

ax1.plot(epochs, history["train_loss"], label="Train", color="steelblue")
ax1.plot(epochs, history["val_loss"],   label="Val",   color="tomato")
ax1.set_title("Loss"); ax1.set_xlabel("Epoch"); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs, history["val_acc"], color="seagreen", linewidth=2)
ax2.axhline(0.5, color="gray", linestyle="--", linewidth=0.8, label="Random baseline")
ax2.set_title("Validation Accuracy"); ax2.set_xlabel("Epoch")
ax2.set_ylim(0, 1); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle("ResNet-18 Transfer Learning — Husky Classifier", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Full evaluation on validation set ─────────────────────────────────────────
model.eval()
all_preds, all_labels_list, all_probs = [], [], []

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        probs = torch.sigmoid(model(imgs).squeeze(1)).cpu()
        preds = (probs >= 0.5).int()
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
        all_labels_list.extend(labels.int().tolist())

# Metrics (reusing logic from notebook 02)
def metrics(y_true, y_pred):
    TP = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 1)
    TN = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 0)
    FP = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 1)
    FN = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 0)
    acc  = (TP + TN) / len(y_true)
    prec = TP / (TP + FP) if (TP + FP) > 0 else 0
    rec  = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    return dict(TP=TP, TN=TN, FP=FP, FN=FN, acc=acc, prec=prec, rec=rec, f1=f1)

m = metrics(all_labels_list, all_preds)

print("── Validation Results ───────────────────────────────")
print(f"  Accuracy:   {m['acc']:.3f}")
print(f"  Precision:  {m['prec']:.3f}")
print(f"  Recall:     {m['rec']:.3f}")
print(f"  F1 Score:   {m['f1']:.3f}")
print(f"\n  TP={m['TP']}  TN={m['TN']}  FP={m['FP']}  FN={m['FN']}")

# Confusion matrix heatmap
import numpy as np
cm = np.array([[m['TN'], m['FP']], [m['FN'], m['TP']]])
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["Pred: Not Husky", "Pred: Husky"])
ax.set_yticklabels(["True: Not Husky", "True: Husky"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=14)
ax.set_title("Confusion Matrix — Validation Set")
plt.tight_layout()
plt.show()

In [ ]:
# ── Visual predictions on validation samples ──────────────────────────────────
def denormalise(tensor):
    """Reverse ImageNet normalisation for display."""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

model.eval()
sample_imgs, sample_labels = next(iter(val_loader))
with torch.no_grad():
    sample_probs = torch.sigmoid(model(sample_imgs.to(device)).squeeze(1)).cpu()
sample_preds = (sample_probs >= 0.5).int()

n_show = 10
fig, axes = plt.subplots(2, 5, figsize=(15, 7))
fig.suptitle("Model Predictions on Validation Images", fontsize=13)

for i, ax in enumerate(axes.flat):
    if i >= n_show:
        break
    img   = denormalise(sample_imgs[i])
    prob  = sample_probs[i].item()
    pred  = sample_preds[i].item()
    true  = sample_labels[i].item()
    correct = pred == true

    ax.imshow(img)
    ax.axis("off")

    label_str = f"{'Husky' if pred == 1 else 'Not Husky'}\n{prob:.0%} confidence"
    color = "green" if correct else "red"
    marker = "✓" if correct else "✗"
    ax.set_title(f"{marker} {label_str}", fontsize=8, color=color)

plt.tight_layout()
plt.show()

print("Green title = correct prediction  |  Red title = wrong prediction")

## What just happened?

We trained a working image classifier with ~300 lines of code and ~150 training images per class. A few things worth noting:

**Why it works so well:**
- ResNet-18 already knows how to see. ImageNet training gave it features for fur texture, eye shape, ear position — all relevant to identifying huskies. We just told it which combination matters.

**Where it will fail:**
- Malamutes, Samoyeds, and other white fluffy dogs look very similar to huskies. The model might confuse them.
- A husky in an unusual pose or lighting might fool it.
- It has only seen ~150 huskies. A breed it's never encountered could produce confident wrong answers.

**The generalisation gap:**
- High validation accuracy on Stanford Dogs images doesn't mean it will work on photos from your phone. The distribution of real-world photos (different cameras, backgrounds, contexts) is different from the dataset.
- This is one of the central challenges in applied ML: a model that performs well in training often degrades in production.

---

**Next section:** NLP/NLU — how models understand and generate text.